In [ ]:
import torch
import torch.nn as nn
from tqdm import tqdm

from red import UNet3D
from qsmloader import QSMLoader
from torch.utils.data import DataLoader
from utils import plot_3d_medical_image
import matplotlib.pyplot as plt
from loss import quantile_regression_loss_fn
from scipy import io
from utils import continuous_dipole_kernel
import numpy as np

In [ ]:
torch.cuda.init()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Load Cosmos

In [ ]:
data = io.loadmat('F:/data/Cosmos_SNR100.mat')
chi_cosmos = data['chi_cosmos'] * data['mask_use']
mask_cosmos = data['mask_use']

data = io.loadmat('F:/data/evaluation_mask.mat')
eval_mask_cosmos = data['evaluation_mask']

In [ ]:

def gen_cosmos(jump=False):
    phase = np.real(np.fft.ifftn(np.fft.fftn(chi_cosmos) * continuous_dipole_kernel(chi_cosmos.shape))) * mask_cosmos

    mag = chi_cosmos - chi_cosmos.min()
    mag = mag / mag.max()
    mag = mag * mask_cosmos

    scale = np.pi / (2 * np.max(np.abs(phase)))
    signal = mag * np.exp(1j * phase * scale)

    snr = np.random.randint(95, 105, (1,))

    signal = signal + ((1. / snr) * (np.random.randn(*signal.shape) + 1j * np.random.randn(*signal.shape)))

    phase = np.angle(signal).astype(np.float32) / scale
    phase = phase * mask_cosmos
    if jump:
        phase[80, 80, 80] = 2
        phase[80, 80, 79] = -2

    p = torch.Tensor(phase.copy()).unsqueeze(0).unsqueeze(0)
    return p

In [ ]:
def eval_cosmos(recon):
    metrics = {}
    metrics['rmse'] = 100 * np.linalg.norm(recon.ravel() - chi_cosmos.ravel()) / np.linalg.norm(chi_cosmos.ravel())
    for i in range(12):
        sub_mask = eval_mask_cosmos == i
        pred = recon[sub_mask]
        gt = chi_cosmos[sub_mask]
        metrics[f'rmse_zona_{i}'] = 100 * np.linalg.norm(gt - pred) / np.linalg.norm(gt)
        metrics[f'mean_zona_{i}'] = np.mean(pred)
        metrics[f'std_zona_{i}'] = np.std(pred)
    return metrics

# Functions for model

In [ ]:
def eval_model(model_in, p, mask):
    model_in.eval()
    with torch.inference_mode():
        out = model_in((p).to(device)).cpu() * mask

    output = out * 1
    lam = 1.09375
    output[:, 0, :, :, :] = torch.minimum(output[:, 0, :, :, :], output[:, 1, :, :, :])
    output[:, 2, :, :, :] = torch.maximum(output[:, 2, :, :, :], output[:, 1, :, :, :])
    upper_edge = lam * (output[:, 2, :, :, :] - output[:, 1, :, :, :]) + output[:, 1, :, :, :]
    prediction = output[:, 1, :, :, :]
    lower_edge = output[:, 1, :, :, :] - lam * (output[:, 1, :, :, :] - output[:, 0, :, :, :])

    upper_edge = upper_edge[0].numpy()
    prediction = prediction[0].numpy()
    lower_edge = lower_edge[0].numpy()
    return upper_edge, prediction, lower_edge

In [ ]:
p = gen_cosmos(True)
p2 = gen_cosmos(False)

# Model incertidumbre Susceptibilidad

In [ ]:
model = UNet3D(in_channels=1, out_channels=3)
model = model.to(device)
model.eval()

In [ ]:
files = [f'./saved/checkpoint_epoch_{i}.pth' for i in range(1, 19)] + \
        [f'./saved/checkpoint_epoch_{i}_respaldo.pth' for i in [1, 8]] + \
        [f'./saved/checkpoint_epoch_8_respaldo2.pth']

In [ ]:
rmse = []
rmse2 = []
for i in tqdm(range(len(files))):
    model.load_state_dict(torch.load(files[i], weights_only=True)['model'])

    model = model.to(device)
    model.eval()
    with torch.inference_mode():
        out = model((p).to(device)).cpu() * mask_cosmos
    gt = chi_cosmos * 1.
    pred = out[0, 1].numpy() * mask_cosmos
    m = mask_cosmos == 1
    rmse.append(100 * np.linalg.norm(pred.ravel() - gt.ravel()) / np.linalg.norm(gt.ravel()))

    with torch.inference_mode():
        out = model((p2).to(device)).cpu() * mask_cosmos
    gt = chi_cosmos * 1.
    pred = out[0, 1].numpy() * mask_cosmos
    m = mask_cosmos == 1
    rmse2.append(100 * np.linalg.norm(pred.ravel() - gt.ravel()) / np.linalg.norm(gt.ravel()))

In [ ]:
print((np.argmin(rmse)), (np.argmin(rmse2)), np.argmin([rmse[i] + rmse2[i] for i in range(len(rmse))]))
plt.figure()
plt.plot(rmse, label='normal')
plt.plot(rmse2, label='phase jump')
plt.legend()
plt.show()

In [ ]:
rmse[16], rmse2[16], rmse[10], rmse2[10]

In [ ]:
files[16], files[16]

In [ ]:
model_susc = UNet3D(in_channels=1, out_channels=3)
model_susc.load_state_dict(torch.load(files[16], weights_only=True)['model'])

# Calibrar

In [ ]:
def gen_bounds(output, lam):
    output[:, 0, :, :, :] = torch.minimum(output[:, 0, :, :, :], output[:, 1, :, :, :])
    output[:, 2, :, :, :] = torch.maximum(output[:, 2, :, :, :], output[:, 1, :, :, :])
    upper_edge = lam * (output[:, 2:3, :, :, :] - output[:, 1:2, :, :, :]) + output[:, 1:2, :, :, :]
    # prediction = output[:, 1, :, :, :]
    lower_edge = output[:, 1:2, :, :, :] - lam * (output[:, 1:2, :, :, :] - output[:, 0:1, :, :, :])
    return lower_edge, upper_edge

In [ ]:
def get_inside_factor(lower_edge, upper_edge, ground_truth, mask):
    factor = (lower_edge[mask] <= ground_truth[mask]) & (ground_truth[mask] <= upper_edge[mask])
    return factor.float().mean()

In [ ]:
def get_outside_factor(lower_edge, upper_edge, ground_truth, mask):
    factor = (ground_truth[mask] < lower_edge[mask]) | (ground_truth[mask] > upper_edge[mask])
    return factor.float().mean()

In [ ]:
ds_calib = QSMLoader(list(range(500)), root='./calib/')
calib_dl = DataLoader(ds_calib, batch_size=10, shuffle=True)

## Susceptibilidad

In [ ]:
model_susc = model_susc.to(device)

In [ ]:
preds = []
gts = []
masks = []
for phase, gt, mask, phase_sr in tqdm(calib_dl):
    model_susc.eval()
    with torch.inference_mode():
        out = model_susc((phase).to(device)).cpu() * mask
    preds.append(out)
    gts.append(gt * mask)
    masks.append(mask)

In [ ]:
model_susc = model_susc.cpu()
torch.cuda.empty_cache()

In [ ]:
preds = torch.cat(preds, dim=0)
gts = torch.cat(gts, dim=0)
masks = torch.cat(masks, dim=0)
preds.shape, gts.shape, masks.shape

In [ ]:
lower_edge, upper_edge = gen_bounds(preds, 1.25)
get_inside_factor(lower_edge, upper_edge, gts, masks == 1)

In [ ]:
lambdas = np.linspace(1.2, 1.25, 21)
factors = []

for lamb in tqdm(lambdas):
    lower_edge, upper_edge = gen_bounds(preds, lamb)
    f = get_inside_factor(lower_edge, upper_edge, gts, masks == 1)
    factors.append(f)


In [ ]:
factors, lambdas

In [ ]:
factors = np.asarray(factors)
factors2 = np.abs(factors - 0.9)

idx = np.argmin(factors2)
print(lambdas[idx], factors[idx])

plt.figure()
plt.plot(lambdas, factors2)
plt.show()

In [ ]:
factors[4], lambdas[4]

In [ ]:
model_susc.calib_param = 1.23

# Figures

In [ ]:
p = gen_cosmos(False)

In [ ]:
def eval_model(model_in, p, mask):
    model_in.eval()
    with torch.inference_mode():
        out = model_in((p).to(device)).cpu() * mask

    output = out * 1
    lam = model_susc.calib_param
    output[:, 0, :, :, :] = torch.minimum(output[:, 0, :, :, :], output[:, 1, :, :, :])
    output[:, 2, :, :, :] = torch.maximum(output[:, 2, :, :, :], output[:, 1, :, :, :])
    upper_edge = lam * (output[:, 2, :, :, :] - output[:, 1, :, :, :]) + output[:, 1, :, :, :]
    prediction = output[:, 1, :, :, :]
    lower_edge = output[:, 1, :, :, :] - lam * (output[:, 1, :, :, :] - output[:, 0, :, :, :])

    upper_edge = upper_edge[0].numpy()
    prediction = prediction[0].numpy()
    lower_edge = lower_edge[0].numpy()
    return upper_edge, prediction, lower_edge

In [ ]:
model_susc = model_susc.to(device)

In [ ]:

model_susc.eval()
with torch.inference_mode():
    out = model_susc((p).to(device)).cpu() * mask_cosmos
gt = chi_cosmos * 1.
pred = out[0, 1].numpy() * mask_cosmos
m = mask_cosmos == 1
print(100 * np.linalg.norm(pred.ravel() - gt.ravel()) / np.linalg.norm(gt.ravel()))

In [ ]:
plot_3d_medical_image(pred, rango=(-0.1, 0.1))

In [ ]:
upper_edge, prediction, lower_edge = eval_model(model_susc, p, mask_cosmos)

In [ ]:
print(100 * np.linalg.norm(prediction.ravel() - gt.ravel()) / np.linalg.norm(gt.ravel()))
print(100 * np.linalg.norm(upper_edge.ravel() - gt.ravel()) / np.linalg.norm(gt.ravel()))
print(100 * np.linalg.norm(lower_edge.ravel() - gt.ravel()) / np.linalg.norm(gt.ravel()))

In [ ]:
p.shape

In [ ]:
io.savemat('cosmos_out.mat',
           {
               'prediction': prediction,
               'gt': gt,
               'mask_cosmos': mask_cosmos,
               'p': p[0, 0].numpy(),
           })

In [ ]:
plot_3d_medical_image(prediction, rango=(-0.1, 0.1))
plot_3d_medical_image(upper_edge, rango=(-0.1, 0.1))
plot_3d_medical_image(lower_edge, rango=(-0.1, 0.1))

In [ ]:
from scipy.ndimage import affine_transform


def rotation_matrix(theta):
    return np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta), np.cos(theta)]
    ])


def rotate_image(image, theta):
    # Matriz de rotación
    rot_mat = rotation_matrix(theta)

    # Calcular el centro de la imagen
    center = np.array(image.shape) / 2

    # Trasladar el origen al centro
    offset = center - np.dot(rot_mat, center)

    # Aplicar la transformación afín
    rotated_image = affine_transform(
        image, rot_mat, offset=offset, order=1  # 'order=1' aplica una interpolación bilineal
    )
    return rotated_image


def plot_3d_medical_image2(image, title=None, cmap='gray', rango=None, show=True, rots=(-90, -90, 90)):
    """
    Plot a 3D medical image in three views: axial, coronal, and sagittal.
    
    Parameters:
    - image: 3D numpy array with shape (D, H, W)
    - title: string, optional title for the entire figure
    - cmap: string, colormap to use for the plots (default is 'gray')
    """
    if rango is None:
        rango = (image.min(), image.max())

    D, H, W = image.shape

    im1 = image[D // 2, :, :]
    im1 = rotate_image(im1, np.radians(rots[0]))

    im2 = image[:, H // 2, :]
    im2 = rotate_image(im2, np.radians(rots[1]))

    im3 = image[:, :, W // 2]
    im3 = rotate_image(im3, np.radians(rots[2]))

    xx, yy = np.where(im1 != 0)
    im1 = im1[np.min(xx):np.max(xx), :]

    xx, yy = np.where(im2 != 0)
    im2 = im2[np.min(xx):np.max(xx), :]

    xx, yy = np.where(im3 != 0)
    im3 = im3[np.min(xx):np.max(xx), :]

    zeros = np.zeros((5, im1.shape[1]))
    im = np.concatenate([zeros, im1, zeros, im2, zeros, im3, zeros], axis=0)

    if show:
        # Create a figure with three subplots
        fig, ax = plt.subplots(1, 1, figsize=(15, 5))
        # Sagittal view (side)
        ax.imshow(im, cmap=cmap, aspect='equal', vmin=rango[0], vmax=rango[1])
        # ax3.set_title('Sagittal View')
        ax.axis('off')
        if title:
            fig.suptitle(title, fontsize=32)
        plt.tight_layout()
        plt.show()
    return im


In [ ]:
im1 = plot_3d_medical_image2(lower_edge, show=False)
im2 = plot_3d_medical_image2(prediction, show=False)
im3 = plot_3d_medical_image2(upper_edge, show=False)

im = np.concatenate([im1, im2, im3], axis=1)

sh = np.asarray((im.shape[1], im.shape[0]))
sh = 30 * sh / sh[1]
plt.figure(figsize=sh)
plt.imshow(im, cmap='gray', aspect='equal', vmin=-0.1, vmax=0.1)
plt.axis('off')
plt.show()

In [ ]:
im = plot_3d_medical_image2(p[0, 0] * mask_cosmos, show=False)

sh = np.asarray((im.shape[1], im.shape[0]))
sh = 30 * sh / sh[1]
plt.figure(figsize=sh)
plt.imshow(im, cmap='gray', aspect='equal', vmin=-0.1, vmax=0.1)
plt.axis('off')
plt.show()

In [ ]:
im = plot_3d_medical_image2(gt * mask_cosmos, show=False)

sh = np.asarray((im.shape[1], im.shape[0]))
sh = 30 * sh / sh[1]
plt.figure(figsize=sh)
plt.imshow(im, cmap='gray', aspect='equal', vmin=-0.1, vmax=0.1)
plt.axis('off')
plt.show()

In [ ]:
um = (upper_edge - lower_edge) * mask_cosmos
um = um - np.min(um[mask_cosmos == 1])
um = um / np.max(um)

im = plot_3d_medical_image2(um * mask_cosmos, show=False)

sh = np.asarray((im.shape[1], im.shape[0]))
sh = 30 * sh / sh[1]
plt.figure(figsize=sh)
plt.imshow(im, cmap='gray', aspect='equal', vmin=0, vmax=0.3)
plt.axis('off')
plt.show()

In [ ]:
results = {}
results['lower bound'] = eval_cosmos(lower_edge)
results['prediction'] = eval_cosmos(prediction)
results['upper bound'] = eval_cosmos(upper_edge)
results['GT'] = eval_cosmos(gt)

In [ ]:
import pandas as pd

df = pd.DataFrame(results)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pandas.plotting import table


def dataframe_to_image(df, filename='dataframe_image.png', figsize=(15, 15), row_colors=['#f1f1f2', '#ffffff']):
    """
    Convierte un DataFrame en una imagen con estilo y la guarda en un archivo.

    Parameters:
    df (pd.DataFrame): El DataFrame a convertir en imagen.
    filename (str): El nombre del archivo donde se guardará la imagen.
    figsize (tuple): El tamaño de la figura de matplotlib (ancho, alto).
    row_colors (list): Lista de colores alternos para las filas.

    Returns:
    None
    """
    # Crear una figura y un eje vacío en matplotlib
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis('off')  # Ocultar el eje

    # Añadir la tabla a la figura con el estilo especificado
    table_obj = table(ax, df, loc='center', cellLoc='center', colWidths=[0.2] * len(df.columns))

    # Añadir colores alternos por fila
    cells = table_obj.get_celld()
    for i in range(len(df)):
        for j in range(len(df.columns)):
            cells[i + 1, j].set_facecolor(row_colors[i % len(row_colors)])
            cells[i + 1, j].set_edgecolor('#c0c0c0')

    # Estilizar la cabecera
    for j in range(len(df.columns)):
        cells[0, j].set_facecolor('#4c72b0')
        cells[0, j].set_text_props(color='white', weight='bold')
        cells[0, j].set_edgecolor('#c0c0c0')

    # # Guardar la figura
    # plt.savefig(filename, bbox_inches='tight', dpi=300)
    # plt.close(fig)
    # print(f'La imagen de la tabla ha sido guardada en {filename}')


dataframe_to_image(df, filename='tabla_con_estilo.png')


In [ ]:
data = io.loadmat('cosmos_ndi.mat')
data.keys()

In [ ]:
data['out_sin_precon'].shape, data['prediction'].shape, data['out_con_precon'].shape

In [ ]:
im1 = plot_3d_medical_image2(data['out_sin_precon'] * mask_cosmos, show=False)
im2 = plot_3d_medical_image2(data['prediction'] * mask_cosmos, show=False)
im3 = plot_3d_medical_image2(data['out_con_precon'] * mask_cosmos, show=False)

im = np.concatenate([im1, im2, im3], axis=1)

sh = np.asarray((im.shape[1], im.shape[0]))
sh = 30 * sh / sh[1]
plt.figure(figsize=sh)
plt.imshow(im, cmap='gray', aspect='equal', vmin=-0.1, vmax=0.1)
plt.axis('off')
plt.show()

In [ ]:
results = {}
results['NDI'] = eval_cosmos(data['out_sin_precon'] * mask_cosmos)
results['RED'] = eval_cosmos(data['prediction'] * mask_cosmos)
results['Refinamiento NDI'] = eval_cosmos(data['out_con_precon'] * mask_cosmos)
results['GT'] = eval_cosmos(chi_cosmos * mask_cosmos)

In [ ]:
df = pd.DataFrame(results)
dataframe_to_image(df)


# Challenge 2.0

In [ ]:
data = io.loadmat('sim1_data_pro.mat')
data.keys()

In [ ]:
data['phase'].shape

In [ ]:
phs_scale = 2 * np.pi * 42.7747892 * 7

In [ ]:
sim1 = np.zeros((176, 208, 208))
sim_mask = np.zeros((176, 208, 208))
sim1[:164, :205, :205] = data['new_phase1'][:164, :205, :205] * data['mask'][:164, :205, :205] / phs_scale
sim_mask[:164, :205, :205] = data['mask'][:164, :205, :205]


In [ ]:
sim1 = torch.FloatTensor(sim1).unsqueeze(0).unsqueeze(0)
sim1.shape

In [ ]:
model_susc.eval()
with torch.inference_mode():
    out = model_susc(sim1.to(device)).cpu() * sim_mask
gt = data['gtsim1'] * sim_mask[:164, :205, :205]
pred = out[0, 1].numpy()[:164, :205, :205] * sim_mask[:164, :205, :205]
m = sim_mask[:164, :205, :205] == 1
print(100 * np.linalg.norm(pred[m] - gt[m]) / np.linalg.norm(gt[m]))

In [ ]:
upper_edge, prediction, lower_edge = eval_model(model_susc, sim1, sim_mask)

In [ ]:
upper_edge, prediction, lower_edge = upper_edge[:164, :205, :205], prediction[:164, :205, :205], lower_edge[:164, :205,
                                                                                                 :205]

In [ ]:
print(100 * np.linalg.norm(lower_edge[m] - gt[m]) / np.linalg.norm(gt[m]))
print(100 * np.linalg.norm(prediction[m] - gt[m]) / np.linalg.norm(gt[m]))
print(100 * np.linalg.norm(upper_edge[m] - gt[m]) / np.linalg.norm(gt[m]))

In [ ]:
im1 = plot_3d_medical_image2(lower_edge, show=False, rots=(-90, -90, -90))
im2 = plot_3d_medical_image2(prediction, show=False, rots=(-90, -90, -90))
im3 = plot_3d_medical_image2(upper_edge, show=False, rots=(-90, -90, -90))
im4 = plot_3d_medical_image2(gt, show=False, rots=(-90, -90, -90))

im = np.concatenate([im1, im2, im3, im4], axis=1)

sh = np.asarray((im.shape[1], im.shape[0]))
sh = 30 * sh / sh[1]
plt.figure(figsize=sh)
plt.imshow(im, cmap='gray', aspect='equal', vmin=-0.1, vmax=0.1)
plt.axis('off')
plt.show()

In [ ]:
plot_3d_medical_image(prediction, rango=(-0.1, 0.1))

In [ ]:
um = (upper_edge - lower_edge) * sim_mask[:164, :205, :205]
um = um - np.min(um[sim_mask[:164, :205, :205] == 1])
um = (um / np.max(um)) * sim_mask[:164, :205, :205]

im = plot_3d_medical_image2(um, show=False, rots=(-90, -90, -90))

sh = np.asarray((im.shape[1], im.shape[0]))
sh = 30 * sh / sh[1]
plt.figure(figsize=sh)
plt.imshow(im, cmap='gray', aspect='equal', vmin=0, vmax=0.3)
plt.axis('off')
plt.show()

In [ ]:

io.savemat('ch2_out.mat',
           {
               'prediction': prediction,
               'gt': gt,
               'sim_mask': sim_mask[:164, :205, :205],
               'sim1': sim1[0, 0].numpy(),
           })

In [ ]:
data = io.loadmat('sim1_ndi.mat')
data.keys()

In [ ]:
data['out_sin_precon'].shape, data['prediction'].shape, data['out_con_precon'].shape

In [ ]:


im1 = plot_3d_medical_image2(data['out_sin_precon'] * sim_mask[:164, :205, :205], show=False, rots=(-90, -90, -90))
im2 = plot_3d_medical_image2(data['prediction'] * sim_mask[:164, :205, :205], show=False, rots=(-90, -90, -90))
im3 = plot_3d_medical_image2(data['out_con_precon'] * sim_mask[:164, :205, :205], show=False, rots=(-90, -90, -90))
im4 = plot_3d_medical_image2(gt * sim_mask[:164, :205, :205], show=False, rots=(-90, -90, -90))

im = np.concatenate([im1, im2, im3, im4], axis=1)

sh = np.asarray((im.shape[1], im.shape[0]))
sh = 30 * sh / sh[1]
plt.figure(figsize=sh)
plt.imshow(im, cmap='gray', aspect='equal', vmin=-0.1, vmax=0.1)
plt.axis('off')
plt.show()

In [ ]:
print(100 * np.linalg.norm(data['out_sin_precon'][m] - gt[m]) / np.linalg.norm(gt[m]))
print(100 * np.linalg.norm(data['prediction'][m] - gt[m]) / np.linalg.norm(gt[m]))
print(100 * np.linalg.norm(data['out_con_precon'][m] - gt[m]) / np.linalg.norm(gt[m]))

# Challenge sim2

In [ ]:
phs_scale = 2 * np.pi * 42.7747892 * 7


In [ ]:
data = io.loadmat('./matlab_files/ReleaseDraftChallenge/sim2SNR1_myV.mat')
data.keys()

In [ ]:
sim2 = np.zeros((176, 208, 208))
sim2_mask = np.zeros((176, 208, 208))
sim2[:164, :205, :205] = data['new_phase2'] * data['mask'] / phs_scale
sim2_mask[:164, :205, :205] = data['mask']


In [ ]:
sim2 = torch.FloatTensor(sim2).unsqueeze(0).unsqueeze(0)
sim2.shape

In [ ]:
model_susc = model_susc.to(device)
model_susc.eval()
with torch.inference_mode():
    out = model_susc(sim2.to(device)).cpu() * sim2_mask
gt = data['gtsim2'] * sim2_mask[:164, :205, :205]
pred = out[0, 1].numpy()[:164, :205, :205] * sim2_mask[:164, :205, :205]
m = sim2_mask[:164, :205, :205] == 1
print(100 * np.linalg.norm(pred[m] - gt[m]) / np.linalg.norm(gt[m]))

In [ ]:
upper_edge, prediction, lower_edge = eval_model(model_susc, sim2, sim2_mask)
upper_edge, prediction, lower_edge = upper_edge[:164, :205, :205], prediction[:164, :205, :205], lower_edge[:164, :205,
                                                                                                 :205]


In [ ]:
print(np.round(100 * np.linalg.norm(lower_edge[m] - gt[m]) / np.linalg.norm(gt[m]), 2))
print(np.round(100 * np.linalg.norm(prediction[m] - gt[m]) / np.linalg.norm(gt[m]), 2))
print(np.round(100 * np.linalg.norm(upper_edge[m] - gt[m]) / np.linalg.norm(gt[m]), 2))

In [ ]:
io.savemat('./matlab_files/ReleaseDraftChallenge/ours_recon.mat', {'ours': prediction})

In [ ]:
im1 = plot_3d_medical_image2(lower_edge, show=False, rots=(-90, -90, -90))
im2 = plot_3d_medical_image2(prediction, show=False, rots=(-90, -90, -90))
im3 = plot_3d_medical_image2(upper_edge, show=False, rots=(-90, -90, -90))
im4 = plot_3d_medical_image2(gt, show=False, rots=(-90, -90, -90))

im = np.concatenate([im1, im2, im3, im4], axis=1)

sh = np.asarray((im.shape[1], im.shape[0]))
sh = 30 * sh / sh[1]
plt.figure(figsize=sh)
plt.imshow(im, cmap='gray', aspect='equal', vmin=-0.1, vmax=0.1)
plt.axis('off')
plt.show()

In [ ]:
im1 = plot_3d_medical_image2(lower_edge - gt, show=False, rots=(-90, -90, -90))
im2 = plot_3d_medical_image2(prediction - gt, show=False, rots=(-90, -90, -90))
im3 = plot_3d_medical_image2(upper_edge - gt, show=False, rots=(-90, -90, -90))
im4 = plot_3d_medical_image2(gt, show=False, rots=(-90, -90, -90))

im = np.concatenate([im1, im2, im3, im4], axis=1)

sh = np.asarray((im.shape[1], im.shape[0]))
sh = 30 * sh / sh[1]
plt.figure(figsize=sh)
plt.imshow(im, cmap='gray', aspect='equal', vmin=-0.1 / 2, vmax=0.1 / 2)
plt.axis('off')
plt.show()

In [ ]:
np.min(data['quality']), np.max(data['quality'])

In [ ]:
qa = data['quality'] * sim2_mask[:164, :205, :205]
dp1 = data['dp1'] * sim2_mask[:164, :205, :205]

qa = qa - np.quantile(qa[sim2_mask[:164, :205, :205] == 1], 0.000)
qa = (qa / np.quantile(qa[sim2_mask[:164, :205, :205] == 1], 1)) * sim2_mask[:164, :205, :205]
qa = (1 - qa) * sim2_mask[:164, :205, :205]

dp1 = dp1 - np.quantile(dp1[sim2_mask[:164, :205, :205] == 1], 0)
dp1 = (dp1 / np.quantile(dp1[sim2_mask[:164, :205, :205] == 1], 1)) * sim2_mask[:164, :205, :205]

error = np.abs(prediction - gt)

error = error - np.quantile(error[sim2_mask[:164, :205, :205] == 1], 0.00)
error = (error / np.quantile(error[sim2_mask[:164, :205, :205] == 1], 1)) * sim2_mask[:164, :205, :205]


In [ ]:
um = (upper_edge - lower_edge) * sim2_mask[:164, :205, :205]
um = um - np.quantile(um[sim2_mask[:164, :205, :205] == 1], 0.00)
um = (um / np.quantile(um[sim2_mask[:164, :205, :205] == 1], 1)) * sim2_mask[:164, :205, :205]

im = plot_3d_medical_image2(um, show=False, rots=(-90, -90, -90))
im2 = plot_3d_medical_image2(dp1, show=False, rots=(-90, -90, -90))
im3 = plot_3d_medical_image2(qa, show=False, rots=(-90, -90, -90))
im4 = plot_3d_medical_image2(error, show=False, rots=(-90, -90, -90))

im = np.concatenate([im2, im3, im4, im], axis=1)

sh = np.asarray((im.shape[1], im.shape[0]))
sh = 30 * sh / sh[1]
plt.figure(figsize=sh)
plt.imshow(im, cmap='gray', aspect='equal', vmin=0, vmax=1)
plt.axis('off')
plt.show()

In [ ]:
um = (upper_edge - lower_edge) * sim2_mask[:164, :205, :205]
um = um - np.quantile(um[sim2_mask[:164, :205, :205] == 1], 0.00)
um = (um / np.quantile(um[sim2_mask[:164, :205, :205] == 1], 1)) * sim2_mask[:164, :205, :205]

im = plot_3d_medical_image2(um, show=False, rots=(-90, -90, -90))
im2 = plot_3d_medical_image2(dp1, show=False, rots=(-90, -90, -90))
im3 = plot_3d_medical_image2(qa, show=False, rots=(-90, -90, -90))
im4 = plot_3d_medical_image2(error, show=False, rots=(-90, -90, -90))

im = np.concatenate([im2, im3, im4, im], axis=1)

sh = np.asarray((im.shape[1], im.shape[0]))
sh = 30 * sh / sh[1]
plt.figure(figsize=sh)
plt.imshow(im, cmap='gray', aspect='equal', vmin=0, vmax=1)
plt.axis('off')
plt.show()

In [ ]:
from io import BytesIO
from PIL import Image


def get_im_from_fig(fig):
    buf = BytesIO()
    fig.savefig(buf, format='png')
    buf.seek(0)

    # Paso 3: Leer la imagen desde BytesIO y convertirla en un array de NumPy
    image = Image.open(buf)
    image_array = np.array(image)
    return image_array

In [ ]:
new_um = np.sqrt(qa * um)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(20, 5))
ax[0].hist(dp1[m], bins=100, alpha=0.5, log=True, color='r')
ax[0].set_title('Prior Error Map')
ax[1].hist(qa[m], bins=100, alpha=0.5, log=True, color='b')
ax[1].set_title('1-Quality')
ax[2].hist(um[m], bins=100, alpha=0.5, log=True, color='g')
ax[2].set_title('Uncertainty Map')

# ax[3].hist(new_um[m], bins=100, alpha=0.5, log=True, color=(0, 51 / 255, 51 / 255))
# ax[3].set_title('SQRT( (1-Quality) * (Uncertainty Map) )')

# ax[3].hist(qa[m], bins=100, alpha=0.5, log=True, color='b', label='1-Quality')
# ax[3].hist(dp1[m], bins=100, alpha=0.5, log=True, color='r', label='Prior Error Map')
# ax[3].hist(um[m], bins=100, alpha=0.5, log=True, color='g', label='Uncertainty Map')
# ax[3].legend(loc='upper right')

ax[0].set_xlim([-0.01, 1])
ax[1].set_xlim([-0.01, 1])
ax[2].set_xlim([-0.01, 1])
# ax[3].set_xlim([-0.01, 1])
fig.suptitle('Histograms of Uncertainty Estimations', fontsize=15)
plt.show()

im_fig1 = get_im_from_fig(fig)

In [ ]:
rmse_um = []
rmse_qm = []
rmse_dp1 = []
# rmse_new_um = []

umbrales = np.linspace(0.02, 0.99, 100)

for th in umbrales:
    m2 = um <= th
    rmse_um.append([
        np.round(100 * np.linalg.norm(lower_edge[m & m2] - gt[m & m2]) / np.linalg.norm(gt[m & m2]), 2),
        np.round(100 * np.linalg.norm(prediction[m & m2] - gt[m & m2]) / np.linalg.norm(gt[m & m2]), 2),
        np.round(100 * np.linalg.norm(upper_edge[m & m2] - gt[m & m2]) / np.linalg.norm(gt[m & m2]), 2)
    ])
    m2 = qa <= th
    rmse_qm.append([
        np.round(100 * np.linalg.norm(lower_edge[m & m2] - gt[m & m2]) / np.linalg.norm(gt[m & m2]), 2),
        np.round(100 * np.linalg.norm(prediction[m & m2] - gt[m & m2]) / np.linalg.norm(gt[m & m2]), 2),
        np.round(100 * np.linalg.norm(upper_edge[m & m2] - gt[m & m2]) / np.linalg.norm(gt[m & m2]), 2)
    ])
    m2 = dp1 <= th
    rmse_dp1.append([
        np.round(100 * np.linalg.norm(lower_edge[m & m2] - gt[m & m2]) / np.linalg.norm(gt[m & m2]), 2),
        np.round(100 * np.linalg.norm(prediction[m & m2] - gt[m & m2]) / np.linalg.norm(gt[m & m2]), 2),
        np.round(100 * np.linalg.norm(upper_edge[m & m2] - gt[m & m2]) / np.linalg.norm(gt[m & m2]), 2)
    ])
    # m2 = new_um <= th
    # rmse_new_um.append([
    #     np.round(100 * np.linalg.norm(lower_edge[m & m2] - gt[m & m2]) / np.linalg.norm(gt[m & m2]), 2),
    #     np.round(100 * np.linalg.norm(prediction[m & m2] - gt[m & m2]) / np.linalg.norm(gt[m & m2]), 2),
    #     np.round(100 * np.linalg.norm(upper_edge[m & m2] - gt[m & m2]) / np.linalg.norm(gt[m & m2]), 2)
    # ])

rmse_um = np.asarray(rmse_um)
rmse_qm = np.asarray(rmse_qm)
rmse_dp1 = np.asarray(rmse_dp1)
# rmse_new_um = np.asarray(rmse_new_um)

print('Done')

In [ ]:
final_rmse_pred = 100 * np.linalg.norm(prediction[m] - gt[m]) / np.linalg.norm(gt[m])
final_rmse_lb = 100 * np.linalg.norm(lower_edge[m] - gt[m]) / np.linalg.norm(gt[m])
final_rmse_ub = 100 * np.linalg.norm(upper_edge[m] - gt[m]) / np.linalg.norm(gt[m])

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(20, 5))

ax[0].semilogx([umbrales[0], umbrales[-1]], [final_rmse_lb, final_rmse_lb], 'k--')
ax[0].semilogx(umbrales, rmse_um[:, 0], color='g', label='Uncertainty Estimation')
ax[0].semilogx(umbrales, rmse_qm[:, 0], color='b', label='1-Quality')
ax[0].semilogx(umbrales, rmse_dp1[:, 0], color='r', label='Prior Error Map')
# ax[0].semilogx(umbrales, rmse_new_um[:, 0], color=(51 / 255, 0, 51 / 255),
#                label='SQRT( (1-Quality) * (Uncertainty Map) )')
ax[0].set_xlabel('Threshold')
ax[0].set_ylabel('NRMSE')
ax[0].legend(loc='upper right')
ax[0].set_title('Lower Boundary')

ax[1].semilogx([umbrales[0], umbrales[-1]], [final_rmse_pred, final_rmse_pred], 'k--')
ax[1].semilogx(umbrales, rmse_um[:, 1], color='g', label='Uncertainty Estimation')
ax[1].semilogx(umbrales, rmse_qm[:, 1], color='b', label='1-Quality')
ax[1].semilogx(umbrales, rmse_dp1[:, 1], color='r', label='Prior Error Map')
# ax[1].semilogx(umbrales, rmse_new_um[:, 1], color=(51 / 255, 0, 51 / 255),
#                label='SQRT( (1-Quality) * (Uncertainty Map) )')
ax[1].set_xlabel('Threshold')
ax[1].set_ylabel('NRMSE')
ax[1].legend(loc='upper right')
ax[1].set_title('Punctual Prediction')

ax[2].semilogx([umbrales[0], umbrales[-1]], [final_rmse_ub, final_rmse_ub], 'k--')
ax[2].semilogx(umbrales, rmse_um[:, 2], color='g', label='Uncertainty Estimation')
ax[2].semilogx(umbrales, rmse_qm[:, 2], color='b', label='1-Quality')
ax[2].semilogx(umbrales, rmse_dp1[:, 2], color='r', label='Prior Error Map')
# ax[2].semilogx(umbrales, rmse_new_um[:, 2], color=(51 / 255, 0, 51 / 255),
#                label='SQRT( (1-Quality) * (Uncertainty Map) )')
ax[2].set_xlabel('Threshold')
ax[2].set_ylabel('NRMSE')
ax[2].legend(loc='upper right')
ax[2].set_title('Upper Boundary')
fig.suptitle('NRMSE vs Uncertainty Threshold Mask', fontsize=15)

plt.show()

im_fig2 = get_im_from_fig(fig)

In [ ]:
im_fig = np.concatenate([im_fig1, im_fig2], axis=0)

plt.figure(figsize=(20, 15))
plt.imshow(im_fig[:, 100:-100, ...])
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
from scipy.ndimage import correlate

def compute_correlation_volume(volume1, volume2, window_size):
    """
    Compute the correlation volume between two 3D volumes using a sliding window.
    
    Parameters:
    - volume1: 3D numpy array, first volume
    - volume2: 3D numpy array, second volume
    - window_size: int, size of the sliding window (should be an odd number)
    
    Returns:
    - correlation_volume: 3D numpy array, volume representing the correlation
    """
    assert volume1.shape == volume2.shape, "Volumes must have the same shape"
    
    # Get the dimensions of the volumes
    dims = volume1.shape
    
    # Pad the volumes to handle the edges
    pad_width = window_size // 2
    vol1_padded = np.pad(volume1, pad_width, mode='constant', constant_values=0)
    vol2_padded = np.pad(volume2, pad_width, mode='constant', constant_values=0)
    
    # Create an empty volume for storing the correlation values
    correlation_volume = np.zeros(dims)
    
    # Define a sliding window function to compute local correlation
    def sliding_window_corr(i, j, k):
        win1 = vol1_padded[i:i+window_size, j:j+window_size, k:k+window_size]
        win2 = vol2_padded[i:i+window_size, j:j+window_size, k:k+window_size]
        return np.corrcoef(win1.flatten(), win2.flatten())[0, 1]
    
    # Apply the sliding window function to each voxel
    for i in range(dims[0]):
        for j in range(dims[1]):
            for k in range(dims[2]):
                correlation_volume[i, j, k] = sliding_window_corr(i, j, k)
    
    return correlation_volume

In [ ]:
new_map = compute_correlation_volume(qa, um, 3)

In [ ]:
np.min(im), np.max(im)

In [ ]:

im = plot_3d_medical_image2(new_map*sim2_mask[:164, :205, :205], show=False, rots=(-90, -90, -90))
im[np.isnan(im)] = 0

sh = np.asarray((im.shape[1], im.shape[0]))
sh = 30 * sh / sh[1]
plt.figure(figsize=sh)
plt.imshow(im, cmap='gray', aspect='equal', vmin=0, vmax=1)
plt.axis('off')
plt.show()


In [ ]:
im.shape, im2.shape, im3.shape

In [ ]:

im1 = plot_3d_medical_image2(qa, show=False, rots=(-90, -90, -90))
im2 = plot_3d_medical_image2(um, show=False, rots=(-90, -90, -90))
im3 = plot_3d_medical_image2(new_um, show=False, rots=(-90, -90, -90))

im = np.concatenate([im1, im2, im3], axis=1)

sh = np.asarray((im.shape[1], im.shape[0]))
sh = 30 * sh / sh[1]
im = im[:142, ...]
fig = plt.figure(figsize=sh)
plt.imshow(im, cmap='gray', aspect='equal', vmin=0, vmax=0.5)
plt.axis('off')
plt.show()

im_fig1 = im


In [ ]:

im1 = plot_3d_medical_image2(qa, show=False, rots=(-90, -90, -90))
im2 = plot_3d_medical_image2(um, show=False, rots=(-90, -90, -90))
im3 = plot_3d_medical_image2(new_um, show=False, rots=(-90, -90, -90))

im = np.concatenate([im1, im2, im3], axis=1)
im = (im <= 0.2) & (im != 0)

sh = np.asarray((im.shape[1], im.shape[0]))
sh = 30 * sh / sh[1]
im = im[:142, ...]
fig = plt.figure(figsize=sh)
plt.imshow(im, cmap='gray', aspect='equal', vmin=0, vmax=0.5)
plt.axis('off')
plt.show()

im_fig2 = im


In [ ]:
im_fig = np.concatenate([im_fig1, im_fig2], axis=0)

plt.figure(figsize=(40, 20))
plt.imshow(im_fig, cmap='gray', aspect='equal', vmin=0, vmax=0.5)
plt.axis('off')
plt.tight_layout()
# plt.colorbar()
plt.show()

In [ ]:
m3 = qa <= 0.2
print(np.round(100 * np.linalg.norm(prediction[m & m3] - gt[m & m3]) / np.linalg.norm(gt[m & m3]), 2),
      np.sum(m & m3) / np.sum(m))

m3 = um <= 0.2
print(np.round(100 * np.linalg.norm(prediction[m & m3] - gt[m & m3]) / np.linalg.norm(gt[m & m3]), 2),
      np.sum(m & m3) / np.sum(m))

m3 = new_um <= 0.2
print(np.round(100 * np.linalg.norm(prediction[m & m3] - gt[m & m3]) / np.linalg.norm(gt[m & m3]), 2),
      np.sum(m & m3) / np.sum(m))


In [ ]:
io.savemat('./matlab_files/ReleaseDraftChallenge/recons_sim2.mat', {
    'LB': lower_edge,
    'PP': prediction,
    'UB': upper_edge,
})

# Invivo

In [ ]:
data = io.loadmat('./matlab_files/In_vivo/MSMV/in_vivo_blend2.mat')
data.keys()

In [ ]:
invivo_mask = data['mask_final'] >= 0.5
invivo_phase = data['phase_final'] * invivo_mask
invivo_quality = data['quality'] * invivo_mask

im = plot_3d_medical_image2(invivo_phase, rots=(-90, -90, -90), rango=(-0.06, 0.06))


sh = np.asarray((im.shape[1], im.shape[0]))
sh = 30 * sh / sh[1]
plt.figure(figsize=sh)
plt.imshow(im, cmap='gray', aspect='equal', vmin=-0.06, vmax=0.06)
plt.axis('off')
plt.show()

In [ ]:
np.asarray(invivo_phase.shape), np.asarray(invivo_phase.shape) / 16

In [ ]:
16 * 12, 16 * 15, 16 * 8


In [ ]:
invivo_in = np.zeros((192, 240, 128))
invivo_in[:186, :230, :] = invivo_phase

In [ ]:
invivo_mask2 = np.zeros((192, 240, 128))
invivo_mask2[:186, :230, :] = invivo_mask

In [ ]:
invivo_in = torch.FloatTensor(invivo_in).unsqueeze(0).unsqueeze(0)

In [ ]:
upper_edge, prediction, lower_edge = eval_model(model_susc, invivo_in, invivo_mask2)
upper_edge, prediction, lower_edge = upper_edge[:186, :230, :], prediction[:186, :230, :], lower_edge[:186, :230, :]

In [ ]:
plot_3d_medical_image(prediction, rango=(-0.1, 0.1))

In [ ]:

def plot_3d_medical_image2(image, title=None, cmap='gray', rango=None, show=True, rots=(-90, -90, 90)):
    """
    Plot a 3D medical image in three views: axial, coronal, and sagittal.
    
    Parameters:
    - image: 3D numpy array with shape (D, H, W)
    - title: string, optional title for the entire figure
    - cmap: string, colormap to use for the plots (default is 'gray')
    """
    if rango is None:
        rango = (image.min(), image.max())

    D, H, W = image.shape
    m = max(D, H, W)
    image2 = np.zeros((m, m, m))
    m = m // 2
    image2[m - int(np.floor(D / 2)):m + int(np.ceil(D / 2)), m - int(np.floor(H / 2)):m + int(np.ceil(H / 2)),
    m - int(np.floor(W / 2)):m + int(np.ceil(W / 2))] = image

    image = image2
    D, H, W = image.shape

    im1 = image[D // 2, :, :]
    im1 = rotate_image(im1, np.radians(rots[0]))

    im2 = image[:, H // 2, :]
    im2 = rotate_image(im2, np.radians(rots[1]))

    im3 = image[:, :, W // 2]
    im3 = rotate_image(im3, np.radians(rots[2]))

    xx, yy = np.where(im1 != 0)
    im1 = im1[np.min(xx):np.max(xx), :]

    xx, yy = np.where(im2 != 0)
    im2 = im2[np.min(xx):np.max(xx), :]

    xx, yy = np.where(im3 != 0)
    im3 = im3[np.min(xx):np.max(xx), :]

    im = np.concatenate([im1, im2, im3], axis=0)

    xx, yy = np.where(im != 0)
    im = im[:, max(np.min(yy) - 1, 0):min(np.max(yy) + 1, im.shape[1])]

    if show:
        # Create a figure with three subplots
        fig, ax = plt.subplots(1, 1, figsize=(15, 5))
        # Sagittal view (side)
        ax.imshow(im, cmap=cmap, aspect='equal', vmin=rango[0], vmax=rango[1])
        # ax3.set_title('Sagittal View')
        ax.axis('off')
        if title:
            fig.suptitle(title, fontsize=32)
        plt.tight_layout()
        plt.show()
    return im

In [ ]:
im1 = plot_3d_medical_image2(lower_edge, show=False, rots=(-90, -90, -90))
im2 = plot_3d_medical_image2(prediction, show=False, rots=(-90, -90, -90))
im3 = plot_3d_medical_image2(upper_edge, show=False, rots=(-90, -90, -90))

im = np.concatenate([im1, im2, im3], axis=1)

sh = np.asarray((im.shape[1], im.shape[0]))
sh = 30 * sh / sh[1]
plt.figure(figsize=sh)
plt.imshow(im, cmap='gray', aspect='equal', vmin=-0.1, vmax=0.1)
plt.axis('off')
plt.show()

In [ ]:
qa = invivo_quality * invivo_mask2[:186, :230, :]
qa = qa - np.min(qa[invivo_mask2[:186, :230, :] == 1])
qa = (qa / np.max(qa)) * invivo_mask2[:186, :230, :]
qa = (1 - qa) * invivo_mask2[:186, :230, :]

um = (upper_edge - lower_edge) * invivo_mask2[:186, :230, :]
um = um - np.min(um[invivo_mask2[:186, :230, :] == 1])
um = (um / np.max(um)) * invivo_mask2[:186, :230, :]

im = plot_3d_medical_image2(um, show=False, rots=(-90, -90, -90))
im2 = plot_3d_medical_image2(qa, show=False, rots=(-90, -90, -90))

im3 = plot_3d_medical_image2((qa * um) ** (1 / 2), show=False, rots=(-90, -90, -90))
im = np.concatenate([im2, im], axis=1)

sh = np.asarray((im.shape[1], im.shape[0]))
sh = 30 * sh / sh[1]
plt.figure(figsize=sh)
plt.imshow(im, cmap='gray', aspect='equal', vmin=0, vmax=.5)
plt.axis('off')
plt.show()

In [ ]:
im3.shape


In [ ]:
im = im3

sh = np.asarray((im.shape[1], im.shape[0]))
sh = 30 * sh / sh[1]
plt.figure(figsize=sh)
plt.imshow(im, cmap='gray', aspect='equal', vmin=0, vmax=1)
plt.axis('off')
plt.show()

In [ ]:
torch.max(sim2), torch.min(sim2)

In [ ]:
torch.max(carlos), torch.min(carlos)

# Carlos imgs

In [ ]:
datos = io.loadmat('carlos_img.mat')
datos.keys()

In [ ]:
phs_scale = 14.982129701263580

In [ ]:
carlos = np.zeros((176, 208, 208))
carlos_mask = np.zeros((176, 208, 208))
carlos[:164, :205, :205] = datos['local_pdf_m4'][:164, :205, :205] * datos['mask4'] / phs_scale
carlos_mask[:164, :205, :205] = datos['mask4']


In [ ]:
carlos = torch.FloatTensor(carlos).unsqueeze(0).unsqueeze(0)
carlos.shape

In [ ]:

upper_edge, prediction, lower_edge = eval_model(model_susc, carlos, carlos_mask)
upper_edge, prediction, lower_edge = upper_edge[:164, :205, :205], prediction[:164, :205, :205], lower_edge[:164, :205,
                                                                                                 :205]

In [ ]:
plot_3d_medical_image2(prediction, show=True, rots=(-90, -90, -90), rango=(-0.1, 0.1))

In [ ]:
recons = {}
recons_ims = []

for l in ['local_lbv_m4', 'local_pdf_m4', 'local_vsharp_m4', 'unwrapped_seguetotalphase', 'unwrapped_truelocalphase']:
    carlos = np.zeros((176, 208, 208))
    carlos_mask = np.zeros((176, 208, 208))
    carlos[:164, :205, :205] = datos[l][:164, :205, :205] * datos['mask4'] / phs_scale
    carlos_mask[:164, :205, :205] = datos['mask4']
    carlos = torch.FloatTensor(carlos).unsqueeze(0).unsqueeze(0)
    upper_edge, prediction, lower_edge = eval_model(model_susc, carlos, carlos_mask)
    upper_edge, prediction, lower_edge = upper_edge[:164, :205, :205], prediction[:164, :205, :205], lower_edge[:164, :205,:205]
    im = plot_3d_medical_image2(prediction, show=False, rots=(-90, -90, -90), rango=(-0.1, 0.1))
    recons_ims.append(im)
    recons[l] = prediction.copy()

recons.keys()

In [ ]:
im = np.concatenate(recons_ims, axis=1)
sh = np.asarray((im.shape[1], im.shape[0]))
sh = 30 * sh / sh[1]
plt.figure(figsize=sh)
plt.imshow(im, cmap='gray', aspect='equal', vmin=-0.1, vmax=0.1)
plt.axis('off')
plt.show()

In [ ]:
io.savemat('Carlos_recons_CPNet.mat', recons)

% Invivo3

In [ ]:
data = io.loadmat('phase_use.mat')
data['phase_use'].shape

In [ ]:
for i in range(128, 500):
    if i % 16 == 0:
        print(i)
        break

In [ ]:
inphase = np.zeros([192, 240, 128])
inphase[:186, :230, :128] = data['phase_use']*data['mask']

invivo_mask2 = np.zeros([192, 240, 128])
invivo_mask2[:186, :230, :128] = data['mask']

In [ ]:
io.savemat('phase_use2.mat', {'phase_use2': inphase})


In [ ]:
inphase = torch.FloatTensor(inphase).unsqueeze(0).unsqueeze(0)

In [ ]:
upper_edge, prediction, lower_edge = eval_model(model_susc, inphase, invivo_mask2)


In [ ]:
plot_3d_medical_image(prediction,rango=(-0.1, 0.1))

In [ ]:
io.savemat('./matlab_files/ReleaseDraftChallenge/ours_recon_invivo.mat', {'ours': prediction})
